In [2]:
import pandas as pd
import numpy as np
import gdown
from scipy.sparse import csr_matrix

**1. Pobieranie danych**

In [3]:
ID_TRAIN = "1zjZe-u52WdE4obfOj8c1foCysYuJvfjB"
ID_TEST = "1y2AMXgEhL3NCT-0Z0nydxSJ6hI_wnrYK"

print("Pobieranie zbioru treningowego:")
gdown.download(f"https://drive.google.com/uc?id={ID_TRAIN}", "df_train.csv", quiet=False)

print("Pobieranie zbioru testowego:")
gdown.download(f"https://drive.google.com/uc?id={ID_TEST}", "df_test.csv", quiet=False)

# Wczytanie zbiorów
df_train = pd.read_csv("df_train.csv")
df_test = pd.read_csv("df_test.csv")

print(f"Dane treningowe: {len(df_train)} wierszy, dane testowe: {len(df_test)} wierszy")

Pobieranie zbioru treningowego:


Downloading...
From: https://drive.google.com/uc?id=1zjZe-u52WdE4obfOj8c1foCysYuJvfjB
To: /content/df_train.csv
100%|██████████| 64.5M/64.5M [00:00<00:00, 91.3MB/s]


Pobieranie zbioru testowego:


Downloading...
From: https://drive.google.com/uc?id=1y2AMXgEhL3NCT-0Z0nydxSJ6hI_wnrYK
To: /content/df_test.csv
100%|██████████| 11.7M/11.7M [00:00<00:00, 62.3MB/s]


Dane treningowe: 2076316 wierszy, dane testowe: 377262 wierszy


Jak wyglądają dane?

In [4]:
df_train.head()

,user,item,event,timestamp
0,1001518,94832,click,1581465600
1,2062683,52240,click,1581465600
2,2726272,135586,click,1581465600
3,1666865,4593,click,1581465601
4,129722,69676,click,1581465603


**2. Tworzenie słowników i par bezpośrednich przejść (item_t -> item_{t+1}) dla każdego użytkownika**

In [5]:
unique_items = df_train['item'].unique()
item2idx = {item: idx for idx, item in enumerate(unique_items)}
idx2item = {idx: item for item, idx in item2idx.items()}
n_items = len(unique_items)

train_sorted = df_train.sort_values(by=['user', 'timestamp'])
train_sorted['next_item'] = train_sorted.groupby('user')['item'].shift(-1)
transitions = train_sorted.dropna(subset=['next_item'])

In [6]:
print(f"Liczba produktów: {n_items}")
print(transitions.head())

Liczba produktów: 13317
        user    item  event   timestamp  next_item
839633     4    7525  click  1581823094    65386.0
840230     4   65386  click  1581823257   169493.0
840770     4  169493  click  1581823401   115778.0
840837     4  115778  click  1581823419    23751.0
841195     4   23751  click  1581823517   181168.0


In [7]:
row_indices = transitions['item'].map(item2idx).values
col_indices = transitions['next_item'].map(item2idx).values

Znalezienie par bezpośrednich przejść (item_t -> item_{t+1}) dla każdego użytkownika i indeksy macierzy przejścia.

In [8]:
train_sorted['next_item'] = train_sorted.groupby('user')['item'].shift(-1)
transitions = train_sorted.dropna(subset=['next_item'])

row_indices = transitions['item'].map(item2idx).values
col_indices = transitions['next_item'].map(item2idx).values

**3. Tworzenie macierzy zliczeń przejść i normalizacja wierszy do prawdopodobieństw P(B|A)**

In [9]:
counts_matrix = csr_matrix(
    (np.ones(len(row_indices)), (row_indices, col_indices)),
    shape=(n_items, n_items),
    dtype=np.float32
)

row_sums = np.array(counts_matrix.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0
transition_matrix = counts_matrix.multiply(1.0 / row_sums[:, np.newaxis]).tocsr()

**3. Ostatnia kliknięta oferta z train dla danego użytkownika i historia**

In [10]:
last_train_items = train_sorted.groupby('user')['item'].last().to_dict()

user_train_history = df_train.groupby('user')['item'].apply(set).to_dict()

Globalna lista najpopularniejszych ofert

In [11]:
popular_items_train = df_train['item'].value_counts().index.tolist()

**4. Generowanie rekomendacji**

In [12]:
def recommend_mcrs(user, k=10, filter_seen=True):
    last_item = last_train_items.get(user)
    seen_items = user_train_history.get(user, set()) if filter_seen else set()

    recs = []
    if last_item in item2idx:
        item_idx = item2idx[last_item]
        probs = transition_matrix[item_idx].toarray().flatten()

        if probs.sum() > 0:
            top_indices = np.argsort(probs)[::-1]
            for idx in top_indices:
                candidate = idx2item[idx]
                if candidate not in seen_items:
                    recs.append(candidate)
                if len(recs) == k:
                    return recs

    #dopełnienie do k najpopularniejszymi ofertami z treningowych
    for item in popular_items_train:
        if item not in seen_items and item not in recs:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

**5. Ewaluacja (precision + recall)**

BASELINE:

In [13]:
def recommend_baseline(user, k=10, filter_seen=True):
    seen_items = user_train_history.get(user, set()) if filter_seen else set()
    recs = []
    for item in popular_items_train:
        if item not in seen_items:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

Użytkownicy w zbiorach:

In [14]:
train_users = set(df_train['user'].unique())
test_users = set(df_test['user'].unique())
eval_users = list(train_users.intersection(test_users))

In [20]:
test_true_inter = df_test.groupby('user')['item'].apply(set).to_dict()

def evaluate(model_func, users, k=10):
    precisions = []
    recalls = []

    for user in users:
        actual = test_true_inter.get(user, set())
        if not actual:
            continue

        predicted = model_func(user, k=k)

        hits = len(set(predicted).intersection(actual))
        precisions.append(hits / k)
        recalls.append(hits / len(actual))

    return np.mean(precisions), np.mean(recalls)

Ewaluacja dla Łańcuchów Markova i Baseline

In [21]:
p_base, r_base = evaluate(recommend_baseline, eval_users, k=10)
print(f"--- BASELINE (Popularity) ---")
print(f"Precision@10: {p_base:.4f}")
print(f"Recall@10:    {r_base:.4f}")

p_mcrs, r_mcrs = evaluate(recommend_mcrs, eval_users, k=10)
print(f"\n--- MCRS (Markov Chain) ---")
print(f"Precision@10: {p_mcrs:.4f}")
print(f"Recall@10:    {r_mcrs:.4f}")

--- BASELINE (Popularity) ---
Precision@10: 0.0014
Recall@10:    0.0025

--- MCRS (Markov Chain) ---
Precision@10: 0.0189
Recall@10:    0.0556


## 6. SASRec

Implementacja architektury Self-Attentive Sequential Recommendation. Model jest uczony wyłącznie na chronologicznych sekwencjach treningowych, a ewaluacja używa tego samego zbioru użytkowników oraz definicji Precision@10 i Recall@10 co MCRS.

In [ ]:
from recommendationsystems.sasrec import fit_sasrec, precision_recall_at_k

sasrec, sasrec_losses = fit_sasrec(
    df_train,
    epochs=10,
    batch_size=512,
    max_len=50,
    hidden_dim=64,
    num_blocks=2,
    num_heads=1,
    dropout=0.2,
    seed=42,
)

In [23]:
p_sasrec, r_sasrec, n_sasrec_users = precision_recall_at_k(
    sasrec, df_test, users=eval_users, k=10
)
print(f"--- SASRec ({n_sasrec_users} users) ---")
print(f"Precision@10: {p_sasrec:.4f}")
print(f"Recall@10:    {r_sasrec:.4f}")

--- SASRec (44316 users) ---
Precision@10: 0.0259
Recall@10:    0.0616
